# 04 · Hypergraph, evaluation, export


> **Research prototype — not a medical device.** Nothing produced by these notebooks may be
> used to diagnose, treat, or make any decision about a patient.


## Why a hypergraph

A graph edge joins two nodes. A **hyperedge joins a set**. "The twelve studies
that look like this one" is naturally one relation over thirteen nodes, not
sixty-six pairwise edges that lose the fact they belong together.

Four families of hyperedge are built over the frozen node features:

| group | what it connects | built from |
|---|---|---|
| `knn_fused` | `{i} ∪ kNN(i)` in fused space | features |
| `knn_image` | kNN in image space only | features |
| `knn_text` | kNN in report space only | features |
| `meta_view` | studies sharing PA/AP/... | metadata |
| `meta_demo` | studies sharing (sex, age decade) | metadata, when available |
| `proto_label` | k-means prototypes of **training** positives | train labels only |

The layer is the standard HGNN operator

$$X' = \sigma\left(D_v^{-1/2} H W D_e^{-1} H^\top D_v^{-1/2} X \Theta\right)$$

computed as two sparse mat-muls, never as an N×N matrix.

## Leakage rules, enforced in code

- Labels are read **only** for rows whose split is `train`.
- Validation and test nodes join `proto_label` edges by *feature distance*, never
  by their own label.
- Validation and test nodes do take part in message passing. That is the standard
  transductive HGNN setting and it uses their **features**, not their labels. Set
  `hypergraph.transductive=false` for a strictly inductive comparison.

## Why the backbone is frozen here

The hypergraph is transductive: it needs every node's feature at once, which
cannot be held on a T4 while gradients flow through two transformers. Trained on
frozen features, an epoch is milliseconds — which is what makes it possible to
actually ablate the design instead of asserting it helps.

In [ ]:
import subprocess, sys
print(sys.version)
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip() or "no GPU reported")
except FileNotFoundError:
    print("nvidia-smi not found - you are on CPU. Runtime > Change runtime type > T4 GPU.")
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# --- 1. where results live -------------------------------------------------
# Mounting Drive is strongly recommended: Colab disconnects, and every stage
# here writes a resumable checkpoint. Without Drive you start over.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = '/content/drive/MyDrive/dvlhg'
else:
    RUN_ROOT = '/content/dvlhg'

# --- 2. get the code -------------------------------------------------------
# Pick ONE. 'clone' is easiest once you have pushed this repo to GitHub.
SOURCE = 'clone'        # 'clone' | 'zip' | 'drive'
REPO_URL = 'https://github.com/abelsangeeth/DVL-Hyperparameter-for-Lung-Disease-Diagnosis.git'
ZIP_PATH = '/content/dvl-hypergraph.zip'          # if SOURCE == 'zip'
DRIVE_CODE = '/content/drive/MyDrive/dvl-hypergraph'  # if SOURCE == 'drive'

import os, shutil, subprocess, sys
CODE = '/content/dvl-hypergraph'
if not os.path.exists(CODE):
    if SOURCE == 'clone':
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, CODE], check=True)
    elif SOURCE == 'zip':
        if not os.path.exists(ZIP_PATH):
            from google.colab import files
            up = files.upload()            # choose the zip from scripts/make_colab_zip.py
            ZIP_PATH = '/content/' + next(iter(up))
        shutil.unpack_archive(ZIP_PATH, '/content/')
    elif SOURCE == 'drive':
        shutil.copytree(DRIVE_CODE, CODE)
print('code at', CODE, '| contents:', sorted(os.listdir(CODE))[:8])

# --- 3. dependencies -------------------------------------------------------
# Colab already ships torch/torchvision built for its CUDA - never reinstall them.
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'open_clip_torch>=2.24', 'timm>=0.9.12', 'transformers>=4.35',
                'fastapi', 'uvicorn', 'python-multipart'], check=True)

sys.path.insert(0, os.path.join(CODE, 'src'))
os.chdir(CODE)
os.environ['PYTHONPATH'] = os.path.join(CODE, 'src')
os.environ['RUN_ROOT'] = RUN_ROOT
print('run root ->', RUN_ROOT)

In [ ]:
# Everything below is overridden on the command line, so this cell is the only
# place you need to edit. Values here are tuned for a Colab T4.
CFG = dict(
    subset_size = 12000,     # frontal studies pulled from MIMIC-CXR-JPG
    text_mode   = 'indication',   # see docs/LEAKAGE.md before changing this
    batch_size  = 24,
    epochs      = 12,
    diffusion_epochs = 30,
    synth_per_class  = 500,
)

def dvlhg(command, **overrides):
    """Run a dvlhg subcommand with RUN_ROOT and any overrides applied."""
    import os, shlex, subprocess, sys
    args = [sys.executable, '-m', 'dvlhg.cli', *shlex.split(command)]
    args += ['--set', f"paths.root={os.environ['RUN_ROOT']}"]
    for key, value in overrides.items():
        args += ['--set', f'{key}={value}']
    print('$', ' '.join(args[2:]))
    return subprocess.run(args, check=True)

In [ ]:
SOURCE_DATASET = 'mimic'

### Run evaluation

This extracts node features, builds the hypergraph, trains the HGNN, computes
the ablation table with patient-level bootstrap CIs, checks the served path
against the evaluated one, writes the report and figures, and exports the
serving bundle.

In [ ]:
dvlhg('eval',
      **{'data.source': SOURCE_DATASET, 'text.mode': CFG['text_mode']})

### The ablation table

In [ ]:
import json, os, pandas as pd
metrics = json.load(open(os.path.join(os.environ['RUN_ROOT'], 'reports', 'metrics.json')))

rows = []
for name, entry in metrics['results'].items():
    test = entry['test']
    ci = entry.get('ci', {}).get('auroc_macro')
    rows.append({
        'variant': name,
        'AUROC': round(test['auroc_macro'], 4),
        '95% CI': f"{ci['lo']:.3f}-{ci['hi']:.3f}" if ci else '-',
        'AUPRC': round(test['auprc_macro'], 4),
        'F1': round(test['f1_macro'], 4),
        'accuracy': round(test['accuracy_macro'], 4),
        'exact match': round(test['exact_match'], 4),
    })
display(pd.DataFrame(rows).set_index('variant'))

check = metrics.get('inductive_check')
if check:
    print(f"\nserved (frozen-bank) vs evaluated (transductive) on {check['n']} studies:"
          f" mean |dp| = {check['mean_abs_prob_delta']:.4f},"
          f" AUROC {check['auroc_inductive']:.3f} vs {check['auroc_transductive']:.3f}")

**How to read this.** `fusion_hypergraph` beating `fusion` by less than the
width of its confidence interval is *not* evidence the hypergraph helps. Say so
if that is what you see — a well-reported null result is worth more than an
overstated gain.

The `image_only` vs `text_only` gap is the other thing to look at. If
`text_only` is close to `fusion`, the model is leaning on the report; check
`text.mode` is `indication` and not one of the leaky modes.

### Per-finding results

In [ ]:
best = 'fusion_hypergraph' if 'fusion_hypergraph' in metrics['results'] else 'fusion'
per_class = metrics['results'][best]['test']['per_class']
frame = pd.DataFrame(per_class).T[
    ['auroc', 'auprc', 'accuracy', 'precision', 'recall', 'specificity', 'f1',
     'threshold', 'n', 'n_positive', 'prevalence']].round(3)
display(frame)

for name in per_class:
    ci = metrics['results'][best].get('ci', {}).get(f'auroc::{name}')
    if ci:
        print(f"{name:20s} AUROC {per_class[name]['auroc']:.3f}  (95% CI {ci['lo']:.3f}-{ci['hi']:.3f})")

### Figures

In [ ]:
import os
from IPython.display import Image as ShowImage, display
figures = os.path.join(os.environ['RUN_ROOT'], 'reports', 'figures')
for name in ['ablation.png', 'roc_curves.png', 'pr_curves.png',
             'hyperedge_weights.png', 'training_history.png']:
    path = os.path.join(figures, name)
    if os.path.exists(path):
        print(name); display(ShowImage(filename=path))

### The learned hyperedge weights are a result in themselves

Each group starts at weight 1.0. Where the model ends up tells you which
relations carried information — if `knn_text` ends far above `meta_demo`, the
report-space neighbourhood mattered and the demographic bucket did not. That is
a finding worth a sentence in the write-up.

In [ ]:
print(json.dumps(metrics.get('hyperedge_group_weights', {}), indent=2))
print()
print(json.dumps(metrics.get('hypergraph', {}), indent=2))

### The written report

In [ ]:
import os
from IPython.display import Markdown, display
display(Markdown(open(os.path.join(os.environ['RUN_ROOT'], 'reports', 'report.md'),
                      encoding='utf-8').read()))

### Download the serving bundle

Everything notebook 05 and the standalone API need: the backbone weights, the
HGNN weights, the neighbour bank, the thresholds and the model card.

The bank contains **training and validation** nodes only — no test film is ever
shipped, and a query can never retrieve itself.

In [ ]:
import os, shutil
export = os.path.join(os.environ['RUN_ROOT'], 'export')
print(sorted(os.listdir(export)))
archive = shutil.make_archive('/content/dvlhg_bundle', 'zip', export)
print('size: %.1f MB' % (os.path.getsize(archive) / 1e6))
from google.colab import files
files.download(archive)